# KLOT rainfall **area-over-threshold** workflow — Chicago metro, 3–4 July 2026

**Radar:** KLOT (Chicago / Romeoville, IL WSR-88D)  
**Domain:** Chicago metro, `[-88.8, -87.2, 41.0, 42.4]` (lon/lat)  
**Event:** multi-round convective spell, 3–4 July 2026 (UTC)

## What this workflow computes
For every Level II volume, convert the 0.5° reflectivity field to rain rate with the Marshall–Palmer relation $Z = 200\,R^{1.6}$, then measure the **area of the domain** (km²) whose rain rate exceeds a set of intensity thresholds — the familiar imperial rainfall bins **0.25, 0.5, 1, 2, 3 in/hr** (6.35, 12.7, 25.4, 50.8, 76.2 mm/hr). Sweeping this over the two days gives an **area-over-threshold time series** — a measure of storm *coverage* and *organization*, complementary to a point rain-rate series.

## Method notes
1. **Gate area in polar geometry.** Each 0.5°-sweep gate cell has area $\Delta r \cdot (r\,\Delta\phi)$ (range-gate depth × arc length). These cells tile the sweep without overlap, so summing the areas of gates above a threshold gives the true covered area — no Cartesian gridding/interpolation needed.
2. **Dual-pol QC.** Gates are kept only where ρHV ≥ 0.85 and Z ≥ 5 dBZ (removes non-meteorological echo).
3. **Hail cap.** Reflectivity is capped at **53 dBZ** before the Z→R inversion so hail cores don't inflate the rain estimate. We carry both capped and uncapped areas; the capped area is the defensible rainfall product. (A 53 dBZ cap limits Marshall–Palmer rain rate to ~75 mm/hr, just below the 3 in/hr = 76.2 mm/hr bin — so the capped ≥3 in/hr area is identically zero by construction.)

All reflectivity uses cmweather **`ChaseSpectral`** (0–70 dBZ); maps are georeferenced with Py-ART's own gate lon/lat via `PlateCarree` on a Mercator GeoAxes.

## 1 · Imports

In [ ]:
import os, math, glob, urllib.request, tarfile, re, datetime as dt
import numpy as np
import numpy.ma as ma
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import pyart
import cmweather
from PIL import Image

# keep cartopy's Natural Earth downloads inside the working dir (writable)
os.makedirs("cartopy_cache", exist_ok=True)
cartopy.config["data_dir"] = os.path.abspath("cartopy_cache")

## 2 · Published skill helpers (`nexrad-radar-gcs`)

Inlined verbatim so the notebook is self-contained: GCS data access (`nexrad_gcs_keys`, `download_nexrad`), `read_radar_safe`, `dualpol_gatefilter`, and the corrected Mercator georeferencing helpers (`make_radar_geoaxes`, `plot_ppi_map`, `add_context_features`).

In [ ]:
import math

# --- Google Cloud public NEXRAD Level-II bucket (AWS fallback) ---
NEXRAD_GCS_HOST = "https://gcp-public-data-nexrad-l2.storage.googleapis.com"
# Marshall-Palmer Z-R relation: Z = A * R**B  (Z in linear mm^6 m^-3)
MARSHALL_PALMER_A = 200.0
MARSHALL_PALMER_B = 1.6


def scan_time(path):
    """Parse the UTC datetime from a NEXRAD key/filename (SITEyyyymmdd_HHMMSS...)."""
    import re, datetime as dt
    m = re.search(r"[A-Z]{4}(\d{8})_(\d{6})", str(path))
    if not m:
        raise ValueError("no NEXRAD timestamp in %r" % path)
    return dt.datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S")


def nexrad_gcs_keys(site, year, month, day, host=None, full_volumes_only=True):
    """List Level-II volume-scan object keys for one radar SITE on one UTC day
    from the Google Cloud public NEXRAD bucket. Returns a sorted list of keys.
    Requires network access to the bucket host (request_network_access)."""
    import urllib.request, xml.etree.ElementTree as ET
    if host is None:
        host = NEXRAD_GCS_HOST
    prefix = "%04d/%02d/%02d/%s/" % (int(year), int(month), int(day), site)
    keys, marker = [], ""
    while True:
        url = "%s/?prefix=%s&marker=%s" % (host, prefix, marker)
        with urllib.request.urlopen(url, timeout=60) as r:
            root = ET.fromstring(r.read())
        contents = root.findall("{*}Contents")
        if not contents:
            break
        for c in contents:
            keys.append(c.find("{*}Key").text)
        trunc = root.find("{*}IsTruncated")
        if trunc is None or (trunc.text or "").lower() != "true":
            break
        marker = keys[-1]
    if full_volumes_only:  # drop supplemental *_MDM message volumes
        keys = [k for k in keys if not k.endswith("_MDM")]
    return sorted(keys)


def download_nexrad(keys, dest_dir="nexrad_scans", host=None):
    """Download NEXRAD keys from the GCS bucket into dest_dir (skips existing).
    Returns local file paths sorted by scan time."""
    import os, urllib.request
    if host is None:
        host = NEXRAD_GCS_HOST
    os.makedirs(dest_dir, exist_ok=True)
    paths = []
    for k in keys:
        local = os.path.join(dest_dir, os.path.basename(k))
        if not os.path.exists(local):
            urllib.request.urlretrieve("%s/%s" % (host, k), local)
        paths.append(local)
    return sorted(paths, key=scan_time)


def read_radar_safe(path):
    """Read a NEXRAD archive with Py-ART; return None on a corrupt/unreadable file
    (some volume scans in the archive are truncated)."""
    import pyart
    try:
        return pyart.io.read_nexrad_archive(path)
    except Exception:
        return None


def setup_cmweather():
    """Register cmweather colormaps and fix the savefig bbox. Call once before plotting.
    figure-style's apply_figure_style() sets savefig.bbox='tight', which COLLAPSES a
    cartopy GeoAxes on save -- this resets it to 'standard'."""
    import cmweather  # noqa: F401  (import registers the colormaps)
    import matplotlib as mpl
    mpl.rcParams["savefig.bbox"] = "standard"
    return True


def dualpol_gatefilter(radar, rhohv_min=0.85, dbz_min=5.0, field="reflectivity"):
    """Standard dual-pol meteorological gate filter: drop low rho_HV, low dBZ, invalid."""
    import pyart
    gf = pyart.filters.GateFilter(radar)
    gf.exclude_below("cross_correlation_ratio", rhohv_min)
    gf.exclude_below(field, dbz_min)
    gf.exclude_invalid(field)
    return gf


def extract_site_rainrate(radar, site_lat, site_lon, radius_km=2.0,
                          rhohv_min=0.85, dbz_min=5.0, zr_a=None, zr_b=None):
    """Mean reflectivity and Marshall-Palmer rain rate over a circular footprint
    around a site, from the lowest sweep. Averages in LINEAR Z then converts.
    Returns dict(mean_dbz, rain_rate_mmph, n_gates)."""
    import numpy as np, numpy.ma as ma
    if zr_a is None:
        zr_a = MARSHALL_PALMER_A
    if zr_b is None:
        zr_b = MARSHALL_PALMER_B
    sweep0 = radar.extract_sweeps([0])
    lats = sweep0.gate_latitude["data"]
    lons = sweep0.gate_longitude["data"]
    dy = (lats - site_lat) * 111.0
    dx = (lons - site_lon) * 111.0 * math.cos(math.radians(site_lat))
    near = np.sqrt(dx ** 2 + dy ** 2) <= radius_km
    dbz = sweep0.fields["reflectivity"]["data"]
    rho = sweep0.fields["cross_correlation_ratio"]["data"]
    good = (near & ~ma.getmaskarray(dbz) & (dbz >= dbz_min)
            & ~ma.getmaskarray(rho) & (rho >= rhohv_min))
    n = int(np.count_nonzero(good))
    if n == 0:
        return {"mean_dbz": float("nan"), "rain_rate_mmph": float("nan"), "n_gates": 0}
    zlin = 10.0 ** (np.asarray(dbz[good]) / 10.0)
    zmean = float(np.mean(zlin))
    return {"mean_dbz": 10.0 * math.log10(zmean),
            "rain_rate_mmph": float((zmean / zr_a) ** (1.0 / zr_b)),
            "n_gates": n}


def make_radar_geoaxes(fig, radar, rect=(0.05, 0.05, 0.9, 0.9), extent=None,
                       tiler=None, tile_zoom=9, tile_alpha=0.5):
    """Create a correctly-georeferenced GeoAxes for a radar PPI.

    Uses a MERCATOR projection. IMPORTANT georeferencing lesson: do NOT use a
    cartopy AzimuthalEquidistant centered on the radar. cartopy's aeqd transform
    is inaccurate at the origin for a non-zero central_latitude -- it maps the
    radar's own lon/lat to roughly (0, +21 km) instead of (0, 0), so the gate
    field (drawn on that projection) ends up shifted ~21 km south of the radar
    marker (which is plotted from true lon/lat). The symptom is a radar triangle
    sitting NORTH of the cone-of-silence blind zone. Mercator is well-behaved
    over a metro-scale extent, and we plot the data using Py-ART's OWN
    gate_longitude/gate_latitude (spherical, correct) via pcolormesh with a
    PlateCarree transform -- so no aeqd round-trip happens anywhere.

    rect   = [left, bottom, width, height] in figure fraction.
    extent = [lon_min, lon_max, lat_min, lat_max] (lon/lat); optional.
    tiler  = a cartopy.io.img_tiles source (e.g. cimgt.OSM(cache=True)) to draw a
             tiled basemap under the radar; requires network access to the tile
             host (for OSM: a/b/c.tile.openstreetmap.org). None = no basemap.
    Returns the GeoAxes. Draw data/markers with transform=ccrs.PlateCarree().
    """
    import cartopy.crs as ccrs
    proj = tiler.crs if tiler is not None else ccrs.Mercator()
    ax = fig.add_axes(list(rect), projection=proj)
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    if tiler is not None:
        try:
            ax.add_image(tiler, tile_zoom, alpha=tile_alpha)
        except Exception:
            pass  # tiles blocked/unavailable -- fall back to a bare basemap
    return ax


def add_context_features(ax, extent, roads=True, airports=True, states=True,
                         label_interstates=True):
    """Overlay interstates, airports, and state borders on a radar GeoAxes.

    extent = [lon_min, lon_max, lat_min, lat_max]. Interstates (level=='Interstate')
    are drawn in red with route labels; beltways/expressways in orange; airports
    as navy diamonds with IATA/abbrev labels. All from Natural Earth 10m cultural
    layers (allowlisted CDN). Draw AFTER the radar pcolormesh so lines sit on top.
    """
    import cartopy.crs as ccrs, cartopy.feature as cfeature
    import cartopy.io.shapereader as shpreader
    from shapely.geometry import box
    pc = ccrs.PlateCarree()
    clip = box(extent[0], extent[2], extent[1], extent[3])
    if states:
        ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="0.4", linewidth=0.6)
    if roads:
        rd = shpreader.Reader(shpreader.natural_earth(resolution="10m",
                              category="cultural", name="roads"))
        labeled = set()
        for rec in rd.records():
            g = rec.geometry
            if g is None or not g.intersects(clip):
                continue
            lvl = rec.attributes.get("level"); typ = rec.attributes.get("type")
            nm = str(rec.attributes.get("name", "")); gi = g.intersection(clip)
            if lvl == "Interstate":
                ax.add_geometries([gi], crs=pc, edgecolor="#b8231f",
                                  facecolor="none", linewidth=2.2, zorder=6)
                if label_interstates and nm and ("I", nm) not in labeled:
                    pt = gi.representative_point()
                    ax.text(pt.x, pt.y, "I-" + nm, fontsize=8, color="#b8231f",
                            fontweight="bold", transform=pc, zorder=9, ha="center",
                            bbox=dict(boxstyle="round,pad=0.1", fc="white",
                                      ec="none", alpha=0.7))
                    labeled.add(("I", nm))
            elif typ == "Beltway" or rec.attributes.get("expressway", 0) == 1:
                ax.add_geometries([gi], crs=pc, edgecolor="#e08214",
                                  facecolor="none", linewidth=1.3, zorder=6)
    if airports:
        ap = shpreader.Reader(shpreader.natural_earth(resolution="10m",
                              category="cultural", name="airports"))
        for rec in ap.records():
            g = rec.geometry
            if g is None or not g.within(clip):
                continue
            ax.plot(g.x, g.y, marker="D", color="#08306b", markersize=6,
                    transform=pc, zorder=8)
            nm = (rec.attributes.get("iata_code") or rec.attributes.get("abbrev")
                  or rec.attributes.get("name", ""))
            ax.annotate(str(nm), xy=(g.x, g.y), xytext=(4, 3),
                        textcoords="offset points", fontsize=7, color="#08306b",
                        fontweight="bold", transform=pc, zorder=8)


def plot_ppi_map(radar, ax, fig, field="reflectivity", sweep=0, extent=None,
                 cmap="ChaseSpectral", vmin=0.0, vmax=70.0, rhohv_min=0.85,
                 dbz_min=5.0, site=None, title="", colorbar_label=None,
                 alpha=0.85, add_colorbar=True, context=False):
    """Plot a dual-pol-filtered PPI field on a cartopy GeoAxes `ax`.

    extent = [lon_min, lon_max, lat_min, lat_max]; site = (lat, lon).
    Call setup_cmweather() once before the first plot.

    GEOREFERENCING (verified correct): the field is drawn with Py-ART's OWN
    radar.gate_longitude / radar.gate_latitude (spherical geometry, matches the
    true radar location to ~6 decimals) via ax.pcolormesh(..., transform=
    ccrs.PlateCarree()). This deliberately does NOT go through
    RadarMapDisplay.plot_ppi_map on a cartopy AzimuthalEquidistant axes: that
    path shifts the gate field ~21 km south of the radar marker because cartopy's
    aeqd transform is inaccurate at the origin for non-zero central_latitude.
    Build `ax` with make_radar_geoaxes(fig, radar, ...) (Mercator). Markers are
    drawn with transform=ccrs.PlateCarree() (lon/lat).

    context=True overlays interstates + airports + state borders via
    add_context_features(). Returns the QuadMesh from pcolormesh.
    """
    import numpy as np
    import cartopy.crs as ccrs
    pc = ccrs.PlateCarree()
    gf = dualpol_gatefilter(radar, rhohv_min, dbz_min, field)
    sl = radar.get_slice(sweep)
    data = radar.get_field(sweep, field)
    excl = gf.gate_excluded[sl]
    data = np.ma.masked_where(excl, data)
    glon = radar.gate_longitude["data"][sl]
    glat = radar.gate_latitude["data"][sl]
    pm = ax.pcolormesh(glon, glat, data, transform=pc, cmap=cmap,
                       vmin=vmin, vmax=vmax, alpha=alpha, shading="auto", zorder=5)
    if extent is not None:
        ax.set_extent(extent, crs=pc)
    if context and extent is not None:
        add_context_features(ax, extent)
    if add_colorbar:
        cb = fig.colorbar(pm, ax=ax, shrink=0.8, pad=0.02)
        cb.set_label(colorbar_label or field)
    if title:
        ax.set_title(title)
    if site is not None:
        ax.plot(site[1], site[0], marker="*", markersize=17, color="black",
                markerfacecolor="yellow", markeredgewidth=1.2, transform=pc, zorder=11)
    ax.plot(radar.longitude["data"][0], radar.latitude["data"][0],
            marker="^", markersize=10, color="black", markerfacecolor="white",
            markeredgewidth=1.2, transform=pc, zorder=11)
    return pm

make_radar_geoaxes2 = make_radar_geoaxes
plot_ppi_map2 = plot_ppi_map
setup_cmweather()

## 3 · Configuration

In [ ]:
K_SITE_ID = "KLOT"
K_YEAR, K_MONTH, K_DAYS = 2026, 7, (3, 4)

atmos_lat, atmos_lon = 41.7018, -87.9963      # Argonne ATMOS site
atmos = (atmos_lat, atmos_lon)
klot_lat, klot_lon = 41.604444, -88.084444    # radar
k_extent = [-88.8, -87.2, 41.0, 42.4]          # Chicago-metro domain

K_ZR_A, K_ZR_B         = 200.0, 1.6            # Marshall-Palmer Z = A R^B
K_RHOHV_MIN, K_DBZ_MIN = 0.85, 5.0             # dual-pol gate filter
K_HAIL_CAP             = 53.0                   # cap dBZ before Z->R

# intensity thresholds (imperial rainfall bins) and mm/hr equivalents
THRESH_IN = [0.25, 0.5, 1.0, 2.0, 3.0]
THRESH_MM = [round(t*25.4, 2) for t in THRESH_IN]

_ktiler = cimgt.OSM(cache=True)
print("thresholds:", dict(zip(THRESH_IN, THRESH_MM)))

## 4 · Download & extract Level II volumes

KLOT volumes arrive as hourly tar archives from the GCS mirror; we extract full-volume `.ar2v` scans (dropping supplemental `_MDM` message volumes) into `klot_scans/`. Idempotent. Requires network access to `gcp-public-data-nexrad-l2.storage.googleapis.com`.

In [ ]:
os.makedirs("klot_tars", exist_ok=True)
os.makedirs("klot_scans", exist_ok=True)
k_allkeys = []
for day in K_DAYS:
    kk = nexrad_gcs_keys(K_SITE_ID, K_YEAR, K_MONTH, day)
    k_allkeys += kk
    print(f"{K_YEAR}-{K_MONTH:02d}-{day:02d}: {len(kk)} hourly tar archives")

host_url = "https://gcp-public-data-nexrad-l2.storage.googleapis.com/"
for i, k in enumerate(k_allkeys):
    tarpath = os.path.join("klot_tars", os.path.basename(k))
    if not os.path.exists(tarpath):
        urllib.request.urlretrieve(host_url + k, tarpath)
    with tarfile.open(tarpath) as tf:
        for m in (m for m in tf.getmembers() if m.isfile()):
            outp = os.path.join("klot_scans", os.path.basename(m.name))
            if not os.path.exists(outp):
                with tf.extractfile(m) as src, open(outp, "wb") as dst:
                    dst.write(src.read())
    if (i+1) % 6 == 0:
        print(f"[{i+1}/{len(k_allkeys)}] tars done", flush=True)

k_allv = sorted([p for p in glob.glob("klot_scans/*") if "_MDM" not in os.path.basename(p)], key=scan_time)
print("FULL volume scans:", len(k_allv))
print("span:", scan_time(k_allv[0]), "->", scan_time(k_allv[-1]))

## 5 · Area-over-threshold engine

`area_over_thresholds` reduces one volume to the domain area (km²) exceeding each threshold, for both hail-capped and uncapped Marshall–Palmer rain rate. Gate-cell areas are computed in polar geometry ($\Delta r \cdot r\,\Delta\phi$) so no regridding is needed.

In [ ]:
THRESH_IN = [0.25, 0.5, 1.0, 2.0, 3.0]
THRESH_MM = [6.35, 12.7, 25.4, 50.8, 76.2]

def area_over_thresholds(radar, extent, thresh_mm=THRESH_MM, rhohv_min=0.85, dbz_min=5.0,
                         hail_cap=53.0, zr_a=200.0, zr_b=1.6):
    """Total domain area (km^2) exceeding each rain-rate threshold, for hail-capped
    and uncapped Marshall-Palmer rain rate, on sweep 0. Per-gate polar cell area
    (r*dphi*dr) tiles the sweep without overlap."""
    sweep0 = radar.extract_sweeps([0])
    lons = sweep0.gate_longitude["data"]; lats = sweep0.gate_latitude["data"]
    rng  = sweep0.range["data"]; az = sweep0.azimuth["data"]
    dbz  = sweep0.fields["reflectivity"]["data"]
    rho  = sweep0.fields["cross_correlation_ratio"]["data"]
    dr = float(np.median(np.diff(rng)))
    daz = np.abs(np.gradient(np.unwrap(np.deg2rad(az))))
    daz = np.clip(daz, 0, np.deg2rad(2.0))
    R2, A2 = np.meshgrid(rng, daz)
    cell_area_km2 = (dr * (R2 * A2)) / 1e6
    lon0,lon1,lat0,lat1 = extent
    indomain = (lons>=lon0)&(lons<=lon1)&(lats>=lat0)&(lats<=lat1)
    good = (indomain & ~ma.getmaskarray(dbz) & (dbz>=dbz_min)
            & ~ma.getmaskarray(rho) & (rho>=rhohv_min))
    g = np.asarray(dbz)
    rr     = (10.0**(g/10.0)               / zr_a)**(1.0/zr_b)
    rr_cap = (10.0**(np.minimum(g,hail_cap)/10.0) / zr_a)**(1.0/zr_b)
    out = {}
    for t_in, t_mm in zip(THRESH_IN, thresh_mm):
        out[f"area_raw_{t_in}"] = float(cell_area_km2[good & (rr    >= t_mm)].sum())
        out[f"area_cap_{t_in}"] = float(cell_area_km2[good & (rr_cap >= t_mm)].sum())
    out["domain_area_km2"] = float(cell_area_km2[good].sum())
    return out

### Validation on a single volume
Run the engine on one scan and print the area table (sanity check before the full loop).

In [ ]:
vpath = k_allv[len(k_allv)//2]
rv = read_radar_safe(vpath)
res = area_over_thresholds(rv, k_extent)
print(scan_time(vpath))
for t in THRESH_IN:
    print(f"  >= {t:>4} in/hr : capped {res[f'area_cap_{t}']:8.1f} km^2 | uncapped {res[f'area_raw_{t}']:8.1f} km^2")
print("  total raining area (>=5 dBZ):", round(res['domain_area_km2'],1), "km^2")

## 6 · Sweep the engine over all volumes → area time series

~610 volumes; a few minutes. Produces `klot_area_threshold_timeseries.csv` with per-threshold area columns (capped + uncapped) plus the total raining area.

In [ ]:
rows, n_bad = [], 0
for i, p in enumerate(k_allv):
    r = read_radar_safe(p)
    if r is None:
        n_bad += 1; continue
    try:
        a = area_over_thresholds(r, k_extent)
        a["time_utc"] = scan_time(p)
        rows.append(a)
    except Exception:
        n_bad += 1
    del r
    if (i+1) % 80 == 0:
        print(f"{i+1}/{len(k_allv)}  bad={n_bad}", flush=True)

adf = pd.DataFrame(rows).sort_values("time_utc").reset_index(drop=True)
adf["tt"] = pd.to_datetime(adf["time_utc"])
cols = ["time_utc"] + [f"area_cap_{t}" for t in THRESH_IN] + [f"area_raw_{t}" for t in THRESH_IN] + ["domain_area_km2","tt"]
adf = adf[cols]
adf.to_csv("klot_area_threshold_timeseries.csv", index=False)
print("rows:", len(adf), "| skipped:", n_bad)
for t in THRESH_IN:
    c = adf[f"area_cap_{t}"]
    print(f">= {t:>4} in/hr : peak capped area {c.max():8.1f} km^2 at {adf.loc[c.idxmax(),'time_utc']}")

## 7 · Area-over-threshold time series

In [ ]:
apply_style = globals().get("apply_figure_style")
if apply_style: apply_style()
plot_thr = [0.25, 0.5, 1.0, 2.0]
labels = {0.25:"\u2265 0.25 in/hr", 0.5:"\u2265 0.5 in/hr", 1.0:"\u2265 1 in/hr", 2.0:"\u2265 2 in/hr"}
cmap = plt.get_cmap("YlGnBu")
colors = {t: cmap(0.30 + 0.62*i/(len(plot_thr)-1)) for i,t in enumerate(plot_thr)}

fig, ax = plt.subplots(figsize=(12, 4.8))
for t in plot_thr:
    ax.plot(adf.tt, adf[f"area_cap_{t}"], color=colors[t], lw=1.6, label=labels[t])
pk1 = adf.loc[adf["area_cap_1.0"].idxmax()]
ax.plot(pk1.tt, pk1["area_cap_1.0"], "o", color="#d62728", ms=6, zorder=6)
ax.annotate(f"peak \u22651 in/hr area\n{pk1['area_cap_1.0']:.0f} km\u00b2  {pk1.tt:%b %d %H:%M}Z",
            xy=(pk1.tt, pk1["area_cap_1.0"]), xytext=(14, 26), textcoords="offset points",
            fontsize=7, color="#d62728", arrowprops=dict(arrowstyle="-", color="#d62728", lw=0.7))
ax.axvline(pd.Timestamp("2026-07-04 00:00"), color="0.6", ls="--", lw=0.8)
ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
ax.set_xlim(adf.tt.min(), adf.tt.max()); ax.set_ylim(bottom=0)
ax.set_xlabel("Time (UTC)"); ax.set_ylabel("Domain area exceeding threshold (km\u00b2)")
ax.set_title("KLOT domain: area of rain over intensity thresholds, 3\u20134 Jul 2026\n"
             "Marshall\u2013Palmer Z=200R$^{1.6}$, hail-capped at 53 dBZ", fontsize=9)
ax.legend(loc="upper left", frameon=False, title="Rain rate", title_fontsize=7)
ax.grid(alpha=0.25); fig.tight_layout()
fig.savefig("klot_area_threshold_timeseries.png", dpi=200)
plt.show()

## 8 · Peak-area intensity-band map

The domain rain-rate field at the timestep of maximum ≥1 in/hr area, discretized into imperial intensity bands over the metro basemap. Yellow star = ATMOS; white triangle = KLOT.

In [ ]:
if apply_style: apply_style()
def sweep_rr_capped_nb(radar, rhohv_min=0.85, dbz_min=5.0, hail_cap=53.0, zr_a=200.0, zr_b=1.6):
    gf = dualpol_gatefilter(radar, rhohv_min, dbz_min, "reflectivity")
    sl = radar.get_slice(0); dbz = radar.get_field(0, "reflectivity")
    g = np.ma.masked_where(gf.gate_excluded[sl], dbz)
    rr = (10.0**(np.ma.minimum(g, hail_cap)/10.0)/zr_a)**(1.0/zr_b)
    return radar.gate_longitude["data"][sl], radar.gate_latitude["data"][sl], rr

pkrow = adf.loc[adf["area_cap_1.0"].idxmax()]
peak_area_path = [p for p in k_allv if scan_time(p)==pkrow.tt.to_pydatetime()][0]
ra = read_radar_safe(peak_area_path)
glon, glat, rr = sweep_rr_capped_nb(ra)
kst = pd.Timestamp(pyart.util.datetime_from_radar(ra).strftime("%Y-%m-%d %H:%M:%S"))

edges_in = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0]; edges_mm = [e*25.4 for e in edges_in]
band_colors = ["#c6dbef", "#6baed6", "#fdae61", "#f03b20", "#7a0177"]
cmap = ListedColormap(band_colors); cmap.set_under((0,0,0,0))
norm = BoundaryNorm(edges_mm, cmap.N)
pc = ccrs.PlateCarree()

fig = plt.figure(figsize=(10, 9))
axM = make_radar_geoaxes2(fig, ra, rect=(0.05,0.05,0.9,0.9), extent=k_extent, tiler=_ktiler, tile_zoom=10, tile_alpha=0.5)
pm = axM.pcolormesh(glon, glat, np.ma.masked_less(rr, edges_mm[0]), transform=pc, cmap=cmap, norm=norm,
                    alpha=0.80, shading="auto", zorder=5)
try: add_context_features(axM, k_extent)
except Exception: pass
axM.plot(atmos[1], atmos[0], marker="*", ms=17, color="black", mfc="yellow", mew=1.2, transform=pc, zorder=11)
axM.plot(klot_lon, klot_lat, marker="^", ms=10, color="black", mfc="white", mew=1.2, transform=pc, zorder=11)
cb = fig.colorbar(pm, ax=axM, shrink=0.8, pad=0.02, ticks=edges_mm)
cb.ax.set_yticklabels([f"{e:g}\n({m:.0f})" for e,m in zip(edges_in, edges_mm)])
cb.set_label("Rain rate  in/hr  (mm/hr)")
axM.set_title(f"KLOT rainfall intensity bands \u2014 peak heavy-rain area\n{kst:%Y-%m-%d %H:%M:%S} UTC   "
              f"(\u22651 in/hr covers {pkrow['area_cap_1.0']:.0f} km\u00b2)", fontsize=9)
fig.savefig("klot_area_threshold_map.png", dpi=150)
plt.show()

## 9 · Stacked intensity-band partition

Rain area split into bands (differences of the cumulative threshold areas), stacked through time to show how the heavy-rain fraction evolves across the convective rounds.

In [ ]:
if apply_style: apply_style()
adf["band_025_05"] = adf["area_cap_0.25"] - adf["area_cap_0.5"]
adf["band_05_1"]   = adf["area_cap_0.5"]  - adf["area_cap_1.0"]
adf["band_1_2"]    = adf["area_cap_1.0"]  - adf["area_cap_2.0"]
adf["band_ge2"]    = adf["area_cap_2.0"]
bands = ["band_025_05","band_05_1","band_1_2","band_ge2"]
blabels = ["0.25\u20130.5 in/hr","0.5\u20131 in/hr","1\u20132 in/hr","\u2265 2 in/hr"]
bcolors = ["#c6dbef","#6baed6","#fdae61","#d7301f"]
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.stackplot(adf.tt, *[adf[b] for b in bands], labels=blabels, colors=bcolors, alpha=0.9)
ax.axvline(pd.Timestamp("2026-07-04 00:00"), color="0.35", ls="--", lw=0.8)
ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
ax.set_xlim(adf.tt.min(), adf.tt.max()); ax.set_ylim(bottom=0)
ax.set_xlabel("Time (UTC)"); ax.set_ylabel("Domain area by intensity band (km\u00b2)")
ax.set_title("KLOT domain: rain-area partitioned by intensity band, 3\u20134 Jul 2026\n"
             "hail-capped Marshall\u2013Palmer; total height = area \u2265 0.25 in/hr", fontsize=9)
ax.legend(loc="upper left", frameon=False, ncol=2, fontsize=7); fig.tight_layout()
fig.savefig("klot_intensity_bands.png", dpi=200)
plt.show()

## 10 · Findings

**Peak areal coverage (hail-capped Marshall–Palmer):**

| threshold | peak area | when (UTC) |
|---|---|---|
| ≥ 0.25 in/hr | 3 430 km² | 2026-07-04 03:16 |
| ≥ 0.5 in/hr | 2 189 km² | 2026-07-04 03:04 |
| ≥ 1 in/hr | 1 230 km² | 2026-07-04 02:45 |
| ≥ 2 in/hr | 371 km² | 2026-07-04 02:45 |
| ≥ 3 in/hr | 0 km² (capped) — 95 km² uncapped | — |
| total raining area (≥5 dBZ) | 18 542 km² | 2026-07-04 22:03 |

**Coverage and intensity peak at different times.** The largest *heavy-rain area* (every bin from ≥0.25 to ≥2 in/hr) occurred during the **overnight round, 02:45–03:16 UTC on 4 July** — a broad, well-organized system covering the metro. The largest *total raining area* (18 542 km², mostly light rain) came later, in the **afternoon round at 22:03 UTC**. This is the value of an area-over-threshold product: it separates a widespread-but-light system from a compact-but-intense one, which a single point or footprint series cannot.

For reference, the companion point-footprint analysis of this same event found the peak *rain rate over the ATMOS site* at 21:29 UTC on 4 July (afternoon round) — i.e. the afternoon round delivered the highest intensity *at that one location*, while the overnight round delivered the largest *area* of heavy rain across the domain.

**Hail-cap sensitivity.** The 53 dBZ cap only affects the extreme bins: the capped ≥3 in/hr area is identically zero (a 53 dBZ cap limits Marshall–Palmer rain rate to ~75 mm/hr, below the 76.2 mm/hr = 3 in/hr threshold), whereas the uncapped field reaches ~95 km² above 3 in/hr at times — those are hail-contaminated cores, not genuine 3-in/hr rainfall. The ≤2 in/hr bins are unaffected by the cap.

### Outputs
| file | description |
|---|---|
| `klot_area_threshold_timeseries.csv` | per-threshold area (capped + uncapped) + total raining area, all volumes |
| `klot_area_threshold_timeseries.png` | area-over-threshold time series |
| `klot_area_threshold_map.png` | intensity-band map at peak heavy-rain area |
| `klot_intensity_bands.png` | stacked intensity-band partition through time |